<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/master/6_clase_BLAST_en_Biopython_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLAST en Biopython

**Nota para Google Colab:**  
- Ejecuta `!pip install biopython` si aún no lo tienes instalado.  
- Las búsquedas BLAST online requieren conexión a internet y pueden tardar varios minutos.

BLAST significa **Basic Local Alignment Search Tool** (Herramienta de Búsqueda de Alineamientos Locales Básicos). Una búsqueda BLAST permite a un investigador comparar una secuencia de proteína o nucleótido de interés (llamada *query*) con una biblioteca o base de datos de secuencias.

Por ejemplo, tras el descubrimiento de un gen previamente desconocido en el ratón, un científico normalmente realizará una búsqueda BLAST del genoma humano para ver si los humanos portan un gen similar; BLAST identificará secuencias en el genoma humano que se asemejan al gen del ratón basándose en la similitud de secuencia.

Biopython proporciona el módulo **Bio.Blast** para trabajar con las operaciones BLAST del NCBI. Puedes ejecutar BLAST tanto en una conexión local como a través de una conexión a Internet.

### Ejecutar consultas contra la versión online de BLAST

Usamos la función `qblast()` del módulo `Bio.Blast.NCBIWWW` para llamar a la versión online de BLAST. Esta tiene tres argumentos no opcionales:

- El **primer argumento** es el programa BLAST a usar para la búsqueda, como una cadena en minúsculas. Actualmente `qblast` solo funciona con **blastn**, **blastp**, **blastx**, **tblastn** y **tblastx**.
- El **segundo argumento** especifica las bases de datos contra las que buscar.
- El **tercer argumento** es una cadena que contiene tu secuencia de consulta (*query*). Esta puede ser la secuencia en sí, la secuencia en formato FASTA, o un identificador como un número GI.

API: https://ncbi.github.io/blast-cloud/dev/api.html

In [ ]:
from Bio.Blast import NCBIWWW
help(NCBIWWW.qblast)

In [ ]:
# Esta celda puede tardar varios minutos en ejecutarse (depende de los servidores del NCBI)
result_handle = NCBIWWW.qblast(
    "blastn",
    "nt",
    """ggtaagtcctctagtacaaacacccccaatattgtgatataattaaa
attatattcatattctgttgccagaaaaaacacttttaggctatattagagccatcttctttgaagcgttgtc"""
)

In [ ]:
# BLAST genera la salida en formato XML y necesitamos parsearla de alguna manera
from Bio.Blast import NCBIXML

blast_records = NCBIXML.parse(result_handle)

Solo puedes recorrer los registros de BLAST **una vez**, así que asegúrate de guardarlos.  
Si tu archivo BLAST es muy grande, podrías tener problemas de memoria al intentar guardarlos todos en una lista.

In [ ]:
blast_records = list(blast_records)

In [ ]:
blast_records

### ¿Qué hay en un registro BLAST?

El **E-value** de BLAST es el número de hits esperados de calidad similar (puntuación) que podrían encontrarse solo por azar.

Cuanto más bajo sea el E-value, más significativo es el alineamiento.

| Rango de E-value | Interpretación Biológica |
| :--- | :--- |
| **< 1e-50** | Homología extremadamente fuerte; secuencias casi idénticas o altamente conservadas. |
| **1e-10 a 1e-50** | Homología clara; es muy probable que compartan ancestro común y función. |
| **1e-4 a 1e-10** | Homología moderada o remota; requiere análisis adicional (ej. dominios o estructura). |
| **> 1e-3** | Similitud probable por azar; no se recomienda inferir homología biológica. |

In [ ]:
E_VALUE_THRESH = 0.00000000001  # Umbral muy estricto
count = 0

for blast_record in blast_records:
    for alignment in blast_record.alignments:
        for hsp in alignment.hsps:
            if hsp.expect < E_VALUE_THRESH:
                count += 1
                print("****Alineamiento****")
                print("secuencia:", alignment.title)
                print("longitud:", alignment.length)
                print(hsp.query[0:75] + "...")
                print(hsp.match[0:75] + "...")
                print(hsp.sbjct[0:75] + "...")
                print()

print(f"Hay {count} secuencias similares en la salida de BLAST")